# Overview
This notebook provides code to analyze methylation of position 4 (1-indexed) of MYC binding sites (CAC**G**TG) and control sites (nnC**G**TG).

Input files from external sources:
- hg19.fa
- new_refseq_exons_171007.bed
    file of exons to exclude
- GSM5064727_08_GT.sorted.bw
    methylation data from: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5064727
Input files generated by other scripts
- myc_control_sample_{i}_matches_nnCGTG_sampled.bed for i in 1,2..10
    generated when running process_mutation_counts.py

Output files:
- myc_control_sample_{i}_nnCGTG_methylation_values.bed for i in 1,2..10
- myc_control_methylation_values.bed
- myc_methylation_values.bed

Also outputs a chart and MannWhitney U test results in the notebook


In [ ]:
! conda install bioconda::ucsc-bigwigtobedgraph -y

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
from pybedtools import BedTool
import pyBigWig
from scipy.stats import mannwhitneyu
import seaborn as sb

'/Users/kylepinheiro/zhu_paper_genomic_analyses'

### Remove broken portion of GSM5064727_08_GT.sorted.bw file. Something in the region of chr1:905988-1014576 causes the file not to be ingested correctly by UCSC tools. If that region is omitted the file seems to be processed without issue.

In [ ]:
# to see error:
# ! bigWigToBedGraph ./GSM5064727_08_GT.sorted.bw ./temp.bed

In [ ]:
original_bigwig = pyBigWig.open("./input/GSM5064727_08_GT.sorted.bw")
# all chromosomes are added in their entirety except chr1, for which we retrive all but the problematic region
for chrom in original_bigwig.chroms().keys():
    if chrom == "chr1":
        ! bigWigToBedGraph -chrom=chr1 -end=905988 ./GSM5064727_08_GT.sorted.bw ./GSE166154_RAW/temp.bed
        ! cat ./temp.bed >> gastric_methylation.bed
        ! bigWigToBedGraph -chrom=chr1 -start=1014596 ./GSM5064727_08_GT.sorted.bw ./temp.bed
    else: 
        ! bigWigToBedGraph -chrom={chrom} ./GSE166154_RAW/GSM5064727_08_GT.sorted.bw ./temp.bed
    
    ! cat ./temp.bed >> ./gastric_methylation.bed

! rm ./temp.bed

# divide all regions in methylation data to single-nucleotide regions
! bedtools makewindows -b ./gastric_methylation.bed -w 1 -i src > gastric_methylation_individ.bed

In [ ]:
# find matches for CACGTG and nnCGTG
! python fastaRegexFinder2.py --fasta  hg19.fa --regex "(?=(CACGTG))" > cacgtg_matches.bed
! python fastaRegexFinder2.py --fasta  hg19.fa --regex "(?=([CAGT]((?<!C)A|[CGT])CGTG))" > nncgtg_matches.bed

exons = BedTool("new_refseq_exons_171007.bed")

for prefix in ["cacgtg", "nncgtg"]:
    # subtract motif matches that are in exons
    matches_no_exons = BedTool(f"{prefix}_matches.bed").intersect(exons, v=True, output=f"{prefix}_matches_no_exons.bed")

In [ ]:
# intersect methylation bedfile with cacgtg matches
methylation = BedTool("gastric_methylation_individ.bed")
methylation.intersect(BedTool("cacgtg_matches_no_exons.bed"), wa=True, wb=True, output="cacgtg_methylated_matches.bed")

In [4]:
# intersect methylation bedfile with each sample set of nncgtg sites
for i in range(1, 11):
    print(i)
    # the files "myc_control_sample_{i}_matches_nnCGTG_sampled.bed" are created when process_mutation_counts_w_sampling.py is run
    methylation.intersect(BedTool(f"myc_control_sample_{i}_matches_nnCGTG_sampled.bed"), wa=True, wb=True, output=f"myc_control_sample_{i}_nnCGTG_methylated_matches.bed")


In [ ]:
# for analysis of overall nncgtg dataset (not sampled)
methylation.intersect(BedTool("nncgtg_matches_no_exons.bed"), wa=True, wb=True, output="nncgtg_methylated_matches.bed")

## Now I need to filter this down to the 4th (one-indexed) position of each motif.

In [9]:
def process_methylated_matches(prefix):
    """From the file of overlaps between methylation data and MYC or control sites,
    further filter to retrieve specifically the methylation values for position 4 (1-indexed).
    """
    filename = f"{prefix}_methylated_matches.bed"
    df = pd.read_csv(filename, sep="\t", header=None)
    df = df.drop(columns=[1, 4, 7, 8])
    # set position of methylation relative to position of MYC/control site match
    # Note: the position column is 1-indexed
    df["position"] = df[2] - df[5]
    # for '-' strand matches, adjust position to be relative to reading from 5' to 3'
    df.loc[df[9] == "-", "position"] = df[6] - df[2] + 1
    
    # keep only those methylation values that are at position 4 of the MYC/control site
    df = df[df["position"] == 4]  # Note again: the values in the position column are 1-indexed
    
    print(f"num methylated positions: {len(df)}")
    
    # output a bedfile of all the methylation values for the positions of interest for this set of sites
    df = df.drop(columns=["position"])
    df.to_csv(f"./intermediate_and_output/{prefix}_methylation_values.bed", sep="\t", index=False, header=None)
    
    return df[3]

In [4]:
def get_statistics(nncgtg_values, cacgtg_values):
    """Output summary statistics and generate a file for use in plotting or further statistical tests.
    """
    print(mannwhitneyu(cacgtg_values, nncgtg_values, alternative="less"))
    print(f"summary:")
    print(nncgtg_values.describe())
    
    new_df = pd.concat([pd.DataFrame({"value": nncgtg_values, "group": "nncgtg"}), pd.DataFrame({"value": cacgtg_values, "group": "cacgtg"})])
    
    print(new_df.groupby("group")["value"].describe())
    
    return new_df

In [5]:
def plot(new_df):
    """Takes a dataframe with two columns 'group' ('nncgtg' or 'cacgtg') and 'value' (0 to 100) and outputs paired boxplots. Each boxplot corresponds to a group.
    Eg. 
        value   column
        83.333  nncgtg
        100.00  nncgtg
        50.00   cacgtg
        0.00    cacgtg
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    # Create the boxplot on the subplot
    sb.boxplot(data=new_df, x="group", y="value", ax=ax)

    # Customize the plot
    ax.set_title("Methylation distributions of Myc sites and control sites (at position 4)")
    ax.set_ylabel("Methylation level")
    
    # Display the plot
    plt.show()

In [10]:
# retrieve methylation values for CACGTG sites, for use in comparing to various controls.
cacgtg_methylation_vals = process_methylated_matches("cacgtg")

# for each sample set of nncgtg sites, retrieve methylation value of position 4
# also compute the mannwhitneyu test, print summary statistics
for i in range(1, 11):
    print(i)
    nncgtg_methylation_vals = process_methylated_matches(f"myc_control_sample_{i}_nnCGTG")
    new_df = get_statistics(nncgtg_methylation_vals, cacgtg_methylation_vals)

# record all similar methylation data for the whole nncgtg dataset (instead of a sample of them)
nncgtg_methylation_vals = process_methylated_matches("nncgtg")
new_df = get_statistics(nncgtg_methylation_vals, cacgtg_methylation_vals)
plot(new_df)

       0       2         3       5       6  9      10  position
0   chr1  133433   57.1429  133430  133436  +  CACGTG         3
1   chr1  133433   57.1429  133430  133436  -  CACGTG         4
2   chr1  533175   94.4444  533171  533177  +  cacgtg         4
3   chr1  533175   94.4444  533171  533177  -  cacgtg         3
4   chr1  603949   66.6667  603946  603952  +  cacgtg         3
5   chr1  603949   66.6667  603946  603952  -  cacgtg         4
6   chr1  604026   61.1111  604023  604029  +  cacgtg         3
7   chr1  604026   61.1111  604023  604029  -  cacgtg         4
8   chr1  604103  100.0000  604100  604106  +  cacgtg         3
9   chr1  604103  100.0000  604100  604106  -  cacgtg         4
10  chr1  604180   42.8571  604177  604183  +  cacgtg         3
11  chr1  604180   42.8571  604177  604183  -  cacgtg         4
12  chr1  604256   85.7143  604253  604259  +  cacgtg         3
13  chr1  604256   85.7143  604253  604259  -  cacgtg         4
14  chr1  604333   40.0000  604330  6043

ValueError: `x` and `y` must be of nonzero size.